<a href="https://colab.research.google.com/github/deepan98raj-dotcom/My_Project/blob/main/RAG_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install Required Libraries

In [ ]:
!pip -q install pypdf openai chromadb

In [ ]:
!pip -q install sentence-transformers

### Using a Free Alternative Embedding Model

Since the OpenAI API experienced a `RateLimitError` due to insufficient credits, we will switch to a free and open-source embedding model. We'll use a `SentenceTransformer` model, specifically `'all-MiniLM-L6-v2'`, which is a popular choice for generating sentence embeddings efficiently.

The `get_embedding` function will be updated to use this new model.

#2. Configure OpenAI API Key

In [ ]:
import os
from getpass import getpass

if not os.environ.get("Mykey"):
   os.environ["Mykey"] = getpass("Enter the OpenAI API Key: ")

   from openai import OpenAI

   client = OpenAI(api_key=os.environ["Mykey"])
   print("OpenAI API Key set successfully.")



# 3. Upload the PDF

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))
print("PDF uploaded successfully.",pdf_path)

Saving Week-3-to-5-Generative-ai.pdf to Week-3-to-5-Generative-ai (1).pdf
PDF uploaded successfully. Week-3-to-5-Generative-ai (1).pdf


#  4. PDF Reader

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

pages = []
for page_number, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    pages.append({
        'page_number': page_number + 1,
        'text': text
    })

text = "\n".join([page['text'] for page in pages])

print('pages:', len(pages))
print('Characters extracted:', len(text))
print('\nPreview:\n')
print(text[:5000])


pages: 26
Characters extracted: 3866

Preview:

Introduction  to 
Generative  AI
introductory level lecture aimed at explaining what Generative AI 
is, how it is used, and how it differs from traditional machine 
learning methods.
Created on by Thaw Zin Toe Presented by: Mohammad Salim @ IT dept. of TIU

2
AI
AI is a computer  system  that can be smart in a way similar to humans, intelligence agents know 
as Abilities
01
Reason
02
Learn
Think & Solve Problems
03
Act Autonomously
Make decisions and take 
actions independently
Improve their knowledge 
and skills from experience
What is 
difference AI vs 
ML
3
What is Generative AI 4
AI ML
Artificial Intelligence Machine Learning
is a discipline Subfield of AI
5
Artificial 
Intelligence
● is a discipline
● Do with theory and methods to build machines
● Think and act like humans
6
Machine 
Learning
● Subfield of AI
● Trains a model from input data
● Gives computer ability to learn without explicit 
programming
Most  common classification o

# 5. Chunking
A simple character-based chunking.In a production system,chunking can be made more sophisticated using paragraphs,section,headings,token counts,or semantic boundaries.

In [ ]:
chunk_size = 500
overlap = 50

chunks = []
start = 0
chunk_id = 0

while start < len(text):
    end = start + chunk_size
    chunk = text[start:end].strip()

    if chunk:
      chunks.append({
        'id':f'chunk-{chunk_id}',
        'text': chunk
    })
    chunk_id += 1
    start += chunk_size - overlap

print('Total chunks:', len(chunks))
print('\nPreview:\n')
print(chunks[0]['text'])

Total chunks: 9

Preview:

Introduction  to 
Generative  AI
introductory level lecture aimed at explaining what Generative AI 
is, how it is used, and how it differs from traditional machine 
learning methods.
Created on by Thaw Zin Toe Presented by: Mohammad Salim @ IT dept. of TIU

2
AI
AI is a computer  system  that can be smart in a way similar to humans, intelligence agents know 
as Abilities
01
Reason
02
Learn
Think & Solve Problems
03
Act Autonomously
Make decisions and take 
actions independently
Improve their kno


# 6. Generate Embeddings

Embeddings convert each of text into a numerical vector that represents its semantic meaning.


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'all-MiniLM-L6-v2' # Switched to a free, open-source model
model = SentenceTransformer(EMBEDDING_MODEL)

def get_embedding(texts, batch_size=100):
  all_embeddings = []

  # The SentenceTransformer model's encode method handles batching internally
  # We'll just pass the texts directly.
  # The input is expected to be a list of strings.
  embeddings = model.encode(texts, show_progress_bar=False, batch_size=batch_size)
  all_embeddings.extend(embeddings)

  return all_embeddings

chunks_texts = [chunk['text'] for chunk in chunks]
chunk_embeddings = get_embedding(chunks_texts)

print('number of embedding:', len(chunk_embeddings))
print('dimension of embedding:', len(chunk_embeddings[0]))
print('First 10 values:', chunk_embeddings[0][:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

number of embedding: 9
dimension of embedding: 384
First 10 values: [-0.07108714 -0.01302882  0.03985951  0.00778693 -0.07181901 -0.02511659
  0.02579923  0.02795452 -0.05669089  0.03998021]


# 7. Store in the vector Database

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_or_create_collection(name='employee_handbook')

existing = collection.get()
if existing['ids']:
  collection.delete(ids=existing['ids'])

collection.add(ids=[chunk['id'] for chunk in chunks],
    embeddings=chunk_embeddings,
    documents=[chunk['text'] for chunk in chunks],
    metadatas=[{'source': chunk['id']} for chunk in chunks])
print('Chunks stored in vector Database:', collection.count())

Chunks stored in vector Database: 9


# 8. User query --> Query Embedding

In [ ]:
question = 'what is machine leaning'

question_embedding = get_embedding([question])[0]

print('Question:',question)
print('Query embedding dimensions:', len(question_embedding))

Question: what is machine leaning
Query embedding dimensions: 384


# 9. Retriever --> Search the Vector Database

In [ ]:
results = collection.query(query_embeddings=[question_embedding],n_results=3)

retrieved_chunks = results['documents'][0]
retrieved_ids = results['metadatas'][0]
distances = results['distances'][0]

print('Retrieved chunks:\n', retrieved_chunks)

for i,(chunk_id,chunk_text,distance) in enumerate (zip(retrieved_ids, retrieved_chunks, distances),start=1):
  print(f'--- Result{i} | {chunk_id} | Distance: {distance:.2f} ---')
  print(chunk_text[:1000])
  print()


Retrieved chunks:
 ['Introduction  to \nGenerative  AI\nintroductory level lecture aimed at explaining what Generative AI \nis, how it is used, and how it differs from traditional machine \nlearning methods.\nCreated on by Thaw Zin Toe Presented by: Mohammad Salim @ IT dept. of TIU\n\n2\nAI\nAI is a computer  system  that can be smart in a way similar to humans, intelligence agents know \nas Abilities\n01\nReason\n02\nLearn\nThink & Solve Problems\n03\nAct Autonomously\nMake decisions and take \nactions independently\nImprove their kno', 'explicit \nprogramming\nMost  common classification of ML  Models\n7\n1. Supervised  Machine  Learning\nTrained On Labelled data.\n2. Unsupervised  Machine  Learning\nTrained On Unlabelled Data\n3. Deep  Learning  — uses artificial \nneural networks to process  more \ncomplex patterns than traditional \nmachine learning.\n\n8\nDeep  Learning  is a Subset of\nMachine Learning.\nIt Uses Artificial  Neural \nNetworks — allowing them to \nprocess  more co

# 10 Prompt + Retrieved Context

now we cimbine the user's question with the relevant chunks returned by the retriever.

this retrieved context becomes part of the input sent to the LLM.


In [ ]:
context = '\n\n--- Retrieved Chunk ---\n\n'.join(retrieved_chunks)

prompt = f'''Answer the user's question using ONLY the retrieved context below.
          if the answer is not present in the context, say that the information is not available in the provided document.
          Retrieved Context:{context}
          User Question:{question}'''.strip()
print(prompt)

Answer the user's question using ONLY the retrieved context below.
          if the answer is not present in the context, say that the information is not available in the provided document.
          Retrieved Context:Introduction  to 
Generative  AI
introductory level lecture aimed at explaining what Generative AI 
is, how it is used, and how it differs from traditional machine 
learning methods.
Created on by Thaw Zin Toe Presented by: Mohammad Salim @ IT dept. of TIU

2
AI
AI is a computer  system  that can be smart in a way similar to humans, intelligence agents know 
as Abilities
01
Reason
02
Learn
Think & Solve Problems
03
Act Autonomously
Make decisions and take 
actions independently
Improve their kno

--- Retrieved Chunk ---

explicit 
programming
Most  common classification of ML  Models
7
1. Supervised  Machine  Learning
Trained On Labelled data.
2. Unsupervised  Machine  Learning
Trained On Unlabelled Data
3. Deep  Learning  — uses artificial 
neural networks to process  mo

# 11. Generate the Answer - OpenAI Responses API

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GENERATION_MODEL = "google/flan-t5-small"

# Load tokenizer and model for a free, instruction-tuned model
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)

# Encode the prompt
inputs = tokenizer(prompt, return_tensors="pt")

# Generate the response
# You can adjust `max_new_tokens` based on desired answer length
outputs = model.generate(**inputs, max_new_tokens=200)

# Decode the generated tokens to get the answer
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Answer:")
print(answer)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Answer:
machine learning
